# Vestibular Lesion Simulations

A catalog of vestibular pathologies and what the model predicts for each.

## Literature on VN resting discharge and afferent contribution

**⚠️ Citation confidence key:**  
✅ High confidence (remember paper well) | ⚠️ Moderate (remember concept, details may differ) | ❓ Needs verification

### VN resting discharge — intrinsic vs afferent

| Claim | Value | Source | Confidence |
|-------|-------|--------|------------|
| VN resting discharge in primates/cats | 80–120 spk/s | Multiple | ✅ |
| VN discharge after ipsilateral deafferentation | ~60–80% remains | Smith & Curthoys (1989) *Brain Res Rev* | ✅ |
| Intrinsic conductances (INaP, HCN/Ih) maintain firing | — | Dieringer (1995) *Prog Neurobiol*; Beraneck et al. (2007) *J Neurosci* | ✅ |
| Afferent contribution ≈ 20–35% of resting discharge | ~30% | de Waele et al. (1994) *Exp Brain Res*; Ris & Godaux (1998) *J Physiol* | ⚠️ |
| Type I VN neurons lose ~30–40% of drive after deafferentation | — | Straka & Dieringer (2004) *Physiol Rev* | ⚠️ |

### Acute spontaneous nystagmus after unilateral loss

| Condition | SPV (monkey) | Source | Confidence |
|-----------|-------------|--------|------------|
| Unilateral labyrinthectomy (neuritis model) | 15–25 deg/s | Fetter & Zee (1988) *JNEN*; Viirre et al. (1988) | ✅ |
| Unilateral VN lesion (infarct model) | >40 deg/s acutely; faster compensation | Ris & Godaux (1998) *J Physiol* | ⚠️ |
| Bilateral labyrinthectomy | 0 nystagmus; oscillopsia | Cohen et al. (1973) | ✅ |
| Recovery TC (static compensation) | 2–7 days primates | Curthoys & Halmagyi (1995) *Ann NY Acad Sci* | ✅ |

### VOR gain reduction

| Condition | VOR gain | Source | Confidence |
|-----------|---------|--------|------------|
| Unilateral UVH (chronic) | ~0.7–0.8 ipsilesional | Halmagyi & Curthoys (1988) *Arch Neurol* | ✅ |
| Bilateral hypofunction | <0.1 | Gillespie & Minor (1999) *Laryngoscope* | ⚠️ |
| VOR gain after adaptation (±lenses) | ±30% from baseline | Miles & Eighmy (1980) *J Neurophysiol* | ✅ |

---
## Model implementation

### Population convention
- **Model LEFT pop** (`x_L`, indices 0:3) codes **rightward** = anatomical **RIGHT VN** type I
- **Model RIGHT pop** (`x_R`, indices 3:6) codes **leftward** = anatomical **LEFT VN** type I
- Net output: `w_est = x_L − x_R`; positive = apparent rightward drive → eye drifts left

### 30/70 afferent/intrinsic split (`f_afferent = 0.30`)
- `b_vs = 100 deg/s`: healthy equilibrium (both afferent + intrinsic drive)
- `b_intrinsic = b_vs × (1 − f_afferent) = 70 deg/s`: residual discharge after deafferentation
- Deafferentation (neuritis): affected pop drops to 70 → net imbalance = 30 deg/s
- VN infarct (complete silence): affected pop drops to 0 → net imbalance = 100 deg/s

### Helper functions
| Function | What it changes |
|----------|----------------|
| `with_uvh(side, canal_gain_frac)` | Canal gains + VS afferent bias (30/70 split) |
| `with_vn_lesion(side)` | Canal gains + VS bias → 0 (complete VN silence) |

---
## What the model can and cannot simulate

| Condition | Simulatable now? | Helper | Notes |
|-----------|-----------------|--------|-------|
| **Unilateral deafferentation (neuritis)** | ✅ Yes | `with_uvh()` | ~30 deg/s net imbalance |
| **Unilateral VN infarct (complete)** | ✅ Yes | `with_vn_lesion()` | 100 deg/s net imbalance |
| **Bilateral hypofunction** | ✅ Yes | manual `canal_gains → 0` | No nystagmus, reduced VOR |
| **Velocity storage ablation (nodulus)** | ✅ Yes | `with_brain(tau_vs=0.5)` | No TC extension |
| **VOR gain reduction (chronic UVH)** | ✅ Yes | `canal_gains × 0.3` | |
| **Head impulse test (vHIT)** | ✅ Yes | Any params | Brief high-accel rotation |
| **VOR gain adaptation** | ✅ Yes | `canal_gains × scale` | Proxy for cerebellar learning |
| **BPPV** | ❌ Not yet | — | Needs otolith→canal coupling |
| **Superior canal dehiscence** | ❌ Not yet | — | Needs canal-specific geometry |
| **Compensation timecourse** | ❌ Not yet | — | Needs plasticity / learning |

---

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation

from oculomotor.sim.simulator import (
    PARAMS_DEFAULT, with_brain, with_sensory, with_uvh, with_vn_lesion, simulate,
    _IDX_VS, _IDX_VS_L, _IDX_VS_R, _IDX_NI, _IDX_VIS, _IDX_SG,
)
from oculomotor.models.sensory_models.sensory_model import (
    N_CANALS, C_slip, C_pos, C_gate,
)
from oculomotor.models.brain_models import saccade_generator as sg_mod
from oculomotor.sim import stimuli as stim_mod
from oculomotor import __version__

print(f'oculomotor {__version__}')
THETA = PARAMS_DEFAULT
print(f'b_vs={THETA.brain.b_vs}  f_afferent={THETA.brain.f_afferent}  tau_vs={THETA.brain.tau_vs}  K_vs={THETA.brain.K_vs}')

%matplotlib widget
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 9

SR  = 200.0
DT  = 1.0 / SR


# ── Shared helpers ────────────────────────────────────────────────────────────

def vs_net(states):
    """Net VS yaw signal x_L - x_R."""
    return (np.array(states.brain[:, _IDX_VS_L]) - np.array(states.brain[:, _IDX_VS_R]))[:, 0]

def eye_vel(states):
    """Left eye yaw velocity (deg/s)."""
    return np.gradient(np.array(states.plant[:, 0]), DT)

def eye_pos(states):
    """Left eye yaw position (deg)."""
    return np.array(states.plant[:, 0])

def extract_burst(states, theta):
    """Recompute yaw burst via vmap."""
    def _at(state):
        x_vis_ = state.sensory[_IDX_VIS]
        e_pd   = C_pos  @ x_vis_
        gvf    = (C_gate @ x_vis_)[0]
        x_ni_  = state.brain[_IDX_NI]
        _, u_b = sg_mod.step(state.brain[_IDX_SG], e_pd, gvf, x_ni_, theta.brain)
        return u_b
    return np.array(jax.vmap(_at)(states))[:, 0]

def slow_phase(t, ev, burst, threshold=10.0, margin_s=0.05):
    """Mask fast phases using burst signal; interpolate SPV."""
    margin_n = max(1, int(margin_s / DT))
    is_fast  = binary_dilation(np.abs(burst) > threshold, structure=np.ones(2*margin_n+1))
    slow     = ~is_fast
    if slow.sum() < 2:
        return ev.copy()
    return np.interp(t, t[slow], ev[slow])

def ax_fmt(ax, ylabel='', xlabel='', ylim=None):
    ax.axhline(0, color='gray', lw=0.4)
    ax.grid(True, alpha=0.2)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8)
    if ylim:   ax.set_ylim(ylim)
    ax.tick_params(labelsize=7)

---
## 1. Unilateral Vestibular Hypofunction (UVH) — Vestibular Neuritis

Reduced canal gain on one side + loss of afferent drive to the ipsilateral VN population.  
`with_uvh(side='left', canal_gain_frac=0.1)` sets:
- `canal_gains[:3] × 0.1`  (left horizontal/anterior/posterior canals near-silent)
- `b_R → b_intrinsic = b_vs × (1 − f_afferent)` = 70 deg/s (afferent drive to model RIGHT pop lost)

**Note on population convention:** Model LEFT pop (x_L) codes rightward = anatomical RIGHT VN type I.  
Left anatomical UVH removes drive from the MODEL RIGHT pop, reducing it from b_vs=100 to b_intrinsic=70.  
Net at rest: x_L − x_R = 100 − 70 = +30 → apparent rightward signal → eye drifts LEFT (slow phase) → nystagmus beats RIGHT (toward intact side). ✓

**Expected:**
- **Spontaneous nystagmus** beating toward intact (right) side, SPV ≈ 20–30 deg/s acutely
- Reduced VOR gain for ipsilesional (leftward) head impulses  
- Corrective catch-up saccade after ipsilesional impulse (positive head impulse test)
- Symmetric OKAN (visual pathway intact)

**Ref:** Halmagyi & Curthoys (1988) *Arch Neurol*; Curthoys & Halmagyi (1995) *Ann NY Acad Sci* ✅

In [ ]:
%%time
# ── Build UVH params with with_uvh() ────────────────────────────────────────
# Left UVH: canal_gains[:3] * 0.1  +  b_R → b_vs * (1 - f_afferent) = 70 deg/s
THETA_UVH = with_uvh(THETA, side='left', canal_gain_frac=0.1)

b_healthy = float(np.array(THETA.brain.b_vs).ravel()[0])
b_intact  = float(np.array(THETA_UVH.brain.b_vs).ravel()[0])   # x_L (model left = intact R VN)
b_lesion  = float(np.array(THETA_UVH.brain.b_vs).ravel()[3])   # x_R (model right = lesioned L VN)
print(f'Healthy:    b_vs = {b_healthy:.0f} deg/s (both pops)')
print(f'UVH left:   b_L = {b_intact:.0f} (intact model-L pop)   b_R = {b_lesion:.0f} (lesioned model-R pop)')
print(f'Expected rest net = {b_intact - b_lesion:.0f} deg/s → rightward → beats RIGHT ✓')

# ── Part A: Spontaneous nystagmus in dark (no head motion) ──────────────────
REST_S = 10.0
t_sp  = jnp.arange(0.0, REST_S, DT)
T_sp  = len(t_sp)

st_sp_h = simulate(THETA,     t_sp,
                   scene_present_array  = jnp.zeros(T_sp),
                   target_present_array = jnp.zeros(T_sp),
                   max_steps=int(REST_S/0.001)+500, return_states=True)
st_sp_u = simulate(THETA_UVH, t_sp,
                   scene_present_array  = jnp.zeros(T_sp),
                   target_present_array = jnp.zeros(T_sp),
                   max_steps=int(REST_S/0.001)+500, return_states=True)

t_sp_np    = np.array(t_sp)
ev_sp_h    = eye_vel(st_sp_h)
ev_sp_u    = eye_vel(st_sp_u)
burst_sp_h = extract_burst(st_sp_h, THETA)
burst_sp_u = extract_burst(st_sp_u, THETA_UVH)
spv_sp_h   = slow_phase(t_sp_np, ev_sp_h, burst_sp_h)
spv_sp_u   = slow_phase(t_sp_np, ev_sp_u, burst_sp_u)

# Steady-state SPV (last 5 s)
mask_ss = t_sp_np > 5.0
spv_ss  = np.mean(spv_sp_u[mask_ss])
print(f'Spontaneous SPV (UVH left, last 5 s): {spv_ss:+.1f} deg/s  '
      f'(negative = leftward slow phase → beats RIGHT ✓)')

# ── Part B: Head impulse test ────────────────────────────────────────────────
HIT_DUR   = 0.15     # s
HIT_V     = 150.0    # deg/s peak
REST_PRE  = 0.5      # s before
REST_POST = 0.8      # s after

def head_impulse_1d(direction=+1, dt=DT):
    """Brief head impulse: half-sine velocity profile."""
    t_total = REST_PRE + HIT_DUR + REST_POST
    t  = np.arange(0, t_total, dt)
    hv = np.zeros_like(t)
    i0 = int(REST_PRE / dt)
    i1 = i0 + int(HIT_DUR / dt)
    n  = i1 - i0
    hv[i0:i1] = direction * HIT_V * np.sin(np.pi * np.arange(n) / n)
    return jnp.array(t), jnp.array(hv.astype(np.float32))

results = {}
for label, theta, directions in [
    ('Healthy',  THETA,     [+1, -1]),
    ('UVH left', THETA_UVH, [+1, -1]),
]:
    results[label] = {}
    for d in directions:
        t, hv1d = head_impulse_1d(direction=d)
        T = len(t)
        st = simulate(theta, t,
                      head_vel_array        = hv1d,
                      scene_present_array   = jnp.zeros(T),
                      target_present_array  = jnp.zeros(T),
                      max_steps=int((REST_PRE+HIT_DUR+REST_POST)/0.001)+200,
                      return_states=True)
        burst = extract_burst(st, theta)
        ev    = eye_vel(st)
        results[label][d] = dict(t=np.array(t), hv=np.array(hv1d), ev=ev, burst=burst)
        del st

print('Done.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Unilateral Vestibular Hypofunction (Left) — Neuritis Model',
             fontsize=11, fontweight='bold')

C_head    = '#555555'
C_healthy = '#2166ac'
C_uvh     = '#d6604d'

# ── Column 0: Spontaneous nystagmus ─────────────────────────────────────────
ax_ev  = axes[0, 0]
ax_spv = axes[1, 0]

ax_ev.plot(t_sp_np, ev_sp_h, color=C_healthy, lw=1.0, alpha=0.5, label='Healthy raw')
ax_ev.plot(t_sp_np, ev_sp_u, color=C_uvh,     lw=1.0, alpha=0.5, label='UVH raw')
ax_fmt(ax_ev, ylabel='Eye vel (deg/s)', ylim=(-60, 60))
ax_ev.legend(fontsize=7)
ax_ev.set_title('Spontaneous nystagmus (dark, no head motion)')

ax_spv.plot(t_sp_np, spv_sp_h, color=C_healthy, lw=1.5, label='Healthy SPV')
ax_spv.plot(t_sp_np, spv_sp_u, color=C_uvh,     lw=1.5, label='UVH SPV')
ax_spv.text(0.98, 0.05, f'SPV={spv_ss:+.1f} deg/s',
            transform=ax_spv.transAxes, ha='right', fontsize=8, color=C_uvh, fontweight='bold')
ax_fmt(ax_spv, ylabel='SPV (deg/s)', xlabel='Time (s)', ylim=(-60, 60))
ax_spv.legend(fontsize=7)
ax_spv.set_title('Slow-phase velocity — beats RIGHT (negative = leftward slow phase)')

# ── Columns 1–2: Head impulse test ──────────────────────────────────────────
dir_labels = {+1: 'Rightward impulse\n(toward intact side)',
              -1: 'Leftward impulse\n(toward lesioned side)'}

for col_i, d in enumerate([+1, -1]):
    ax_v = axes[0, col_i + 1]
    ax_b = axes[1, col_i + 1]

    for label, color in [('Healthy', C_healthy), ('UVH left', C_uvh)]:
        r = results[label][d]
        ax_v.plot(r['t'], -r['ev'],  color=color, lw=1.8, label=f'{label} (−eye vel)')
        ax_b.plot(r['t'],  r['burst'], color=color, lw=1.2, label=label)

    # Head velocity reference
    r_h = results['Healthy'][d]
    ax_v.plot(r_h['t'], r_h['hv'], color=C_head, lw=1.0, ls='--', label='Head vel', alpha=0.7)

    ax_v.axvline(REST_PRE, color='k', lw=0.6, ls='--', alpha=0.4)
    ax_v.axvline(REST_PRE + HIT_DUR, color='k', lw=0.6, ls='--', alpha=0.4)
    ax_b.axvline(REST_PRE, color='k', lw=0.6, ls='--', alpha=0.4)
    ax_b.axvline(REST_PRE + HIT_DUR, color='k', lw=0.6, ls='--', alpha=0.4)

    ax_v.set_title(dir_labels[d], fontsize=9)
    ax_fmt(ax_v, ylabel='−Eye vel / Head vel\n(deg/s)' if col_i == 0 else '')
    ax_fmt(ax_b, ylabel='Burst (deg/s)' if col_i == 0 else '', xlabel='Time (s)')

    if col_i == 0:
        ax_v.legend(fontsize=7, loc='upper left')
        ax_b.legend(fontsize=7, loc='upper left')

    # Gain annotation
    for label, color in [('Healthy', C_healthy), ('UVH left', C_uvh)]:
        r = results[label][d]
        peak_hv = np.abs(r['hv']).max()
        peak_ev = np.abs(r['ev']).max()
        gain    = peak_ev / peak_hv if peak_hv > 0 else 0
        ax_v.text(0.98, 0.05 + (0.15 if label == 'Healthy' else 0),
                  f'{label}: gain≈{gain:.2f}',
                  transform=ax_v.transAxes, ha='right', fontsize=7,
                  color=color, fontweight='bold')

fig.tight_layout()
plt.show()

---
## 2. Unilateral VN Lesion — Acute Spontaneous Nystagmus (Infarct / Ablation)

Complete silencing of one VN population — models VN infarct, surgical ablation, or intratympanic
gentamicin overdose.

`with_vn_lesion(side='left')` sets:
- `canal_gains[:3] → 0` (left canals silent)
- `b_R → 0` (model RIGHT pop completely silent — no afferent, no intrinsic drive)

**Expected:**
- **Strong spontaneous nystagmus** beating RIGHT (toward intact side), SPV >> neuritis
- Net at rest: x_L − x_R = 100 − 0 = +100 → drives strong leftward slow phase
- VOR abolished for leftward impulses; preserved for rightward
- Faster/stronger than UVH neuritis (both afferent AND intrinsic VN drive lost)

**Ref:** Ris & Godaux (1998) *J Physiol*; Newlands & Perachio (1990) *Exp Brain Res* ⚠️

In [ ]:
%%time
# ── Build VN lesion params ────────────────────────────────────────────────────
THETA_VNL = with_vn_lesion(THETA, side='left')

b_vn_l = float(np.array(THETA_VNL.brain.b_vs).ravel()[0])   # model-L pop (intact anatomical R VN)
b_vn_r = float(np.array(THETA_VNL.brain.b_vs).ravel()[3])   # model-R pop (silenced anatomical L VN)
print(f'VN infarct left:  b_L={b_vn_l:.0f}  b_R={b_vn_r:.0f}  net={b_vn_l-b_vn_r:.0f} deg/s')

# ── Spontaneous nystagmus comparison: Healthy, UVH neuritis, VN infarct ─────
t_sp2  = jnp.arange(0.0, REST_S, DT)
T_sp2  = len(t_sp2)

st_sp_vn = simulate(THETA_VNL, t_sp2,
                    scene_present_array  = jnp.zeros(T_sp2),
                    target_present_array = jnp.zeros(T_sp2),
                    max_steps=int(REST_S/0.001)+500, return_states=True)

t_sp2_np  = np.array(t_sp2)
ev_sp_vn  = eye_vel(st_sp_vn)
burst_vn  = extract_burst(st_sp_vn, THETA_VNL)
spv_sp_vn = slow_phase(t_sp2_np, ev_sp_vn, burst_vn)

mask2   = t_sp2_np > 5.0
spv_vn  = np.mean(spv_sp_vn[mask2])
spv_uvh = np.mean(spv_sp_u[t_sp_np > 5.0])

print(f'SPV — UVH neuritis: {spv_uvh:+.1f} deg/s   VN infarct: {spv_vn:+.1f} deg/s')
print(f'Ratio infarct/neuritis: {abs(spv_vn)/max(abs(spv_uvh),1e-3):.1f}×')

print('Done.')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle('Spontaneous Nystagmus: Healthy vs UVH Neuritis vs VN Infarct (left side)',
             fontsize=10, fontweight='bold')

C_healthy = '#2166ac'
C_uvh     = '#e08214'
C_vn      = '#d6604d'

# Raw eye velocity
axes[0].plot(t_sp_np,  ev_sp_h,  color=C_healthy, lw=0.8, alpha=0.4)
axes[0].plot(t_sp_np,  ev_sp_u,  color=C_uvh,     lw=0.8, alpha=0.4)
axes[0].plot(t_sp2_np, ev_sp_vn, color=C_vn,      lw=0.8, alpha=0.4)
ax_fmt(axes[0], ylabel='Eye vel (deg/s)', ylim=(-120, 120))
axes[0].set_title('Raw eye velocity — nystagmus sawtooth')

# SPV overlaid
axes[1].plot(t_sp_np,  spv_sp_h,  color=C_healthy, lw=2.0, label=f'Healthy  SPV≈0 deg/s')
axes[1].plot(t_sp_np,  spv_sp_u,  color=C_uvh,     lw=2.0,
             label=f'UVH neuritis  SPV≈{spv_uvh:+.0f} deg/s  (30% afferent lost)')
axes[1].plot(t_sp2_np, spv_sp_vn, color=C_vn,      lw=2.0,
             label=f'VN infarct    SPV≈{spv_vn:+.0f} deg/s  (100% VN silenced)')

# Annotate: bias imbalance
axes[1].annotate(f'b_net = b_vs − b_intrinsic = {b_healthy:.0f} − {b_lesion:.0f} = {b_healthy-b_lesion:.0f}',
                 xy=(6, spv_uvh), fontsize=7, color=C_uvh,
                 xytext=(6.5, spv_uvh - 8), arrowprops=dict(arrowstyle='->', color=C_uvh, lw=0.8))
axes[1].annotate(f'b_net = b_vs − 0 = {b_healthy:.0f}',
                 xy=(6, spv_vn), fontsize=7, color=C_vn,
                 xytext=(6.5, spv_vn - 8), arrowprops=dict(arrowstyle='->', color=C_vn, lw=0.8))

ax_fmt(axes[1], ylabel='SPV (deg/s)', xlabel='Time (s)', ylim=(-120, 20))
axes[1].legend(fontsize=7, loc='lower right')
axes[1].set_title('Slow-phase velocity — negative = leftward drift → nystagmus beats RIGHT')

fig.tight_layout()
plt.show()

---
## 3. Bilateral Vestibular Hypofunction (BVH)

Both canal arrays have near-zero gain — models ototoxicity (gentamicin), bilateral Menière's, or
bilateral vestibular aplasia.

**Parameters:** `canal_gains → [0.05, …]` (95% loss bilateral)

**Expected:**
- No spontaneous nystagmus
- VOR gain ≈ 0 → oscillopsia during head movement
- Catch-up saccades during sustained rotation (staircase toward orbital limit)
- Normal OKN (visual pathway intact)
- Normal smooth pursuit

In [ ]:
%%time
CANAL_GAIN_BVH = tuple(np.array(THETA.sensory.canal_gains) * 0.05)
THETA_BVH = with_sensory(THETA, canal_gains=CANAL_GAIN_BVH)

# Sustained rotation — compare healthy vs BVH
V_BVH = 30.0; ROT_BVH = 20.0; CST_BVH = 30.0
t_bvh, hv_bvh = stim_mod.rotation_step(V_BVH, rotate_dur=ROT_BVH, coast_dur=CST_BVH, dt=DT)
T_bvh = len(t_bvh)

st_bvh_h = simulate(THETA,     t_bvh, head_vel_array=hv_bvh,
                    scene_present_array=jnp.zeros(T_bvh),
                    target_present_array=jnp.zeros(T_bvh),
                    max_steps=int((ROT_BVH+CST_BVH)/0.001)+500, return_states=True)
st_bvh_p = simulate(THETA_BVH, t_bvh, head_vel_array=hv_bvh,
                    scene_present_array=jnp.zeros(T_bvh),
                    target_present_array=jnp.zeros(T_bvh),
                    max_steps=int((ROT_BVH+CST_BVH)/0.001)+500, return_states=True)

t_np = np.array(t_bvh)
ev_h = eye_vel(st_bvh_h)
ev_p = eye_vel(st_bvh_p)
hv_np = np.array(hv_bvh[:, 0] if hv_bvh.ndim > 1 else hv_bvh)
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
fig.suptitle('Bilateral Vestibular Hypofunction — Sustained Rotation in Dark',
             fontsize=10, fontweight='bold')

axes[0].plot(t_np, hv_np, color='#555555', lw=1.5)
ax_fmt(axes[0], ylabel='Head vel (deg/s)')
axes[0].set_title('Input: head velocity')

axes[1].plot(t_np, -ev_h, color='#2166ac', lw=1.5, label='Healthy')
axes[1].plot(t_np, -ev_p, color='#d6604d', lw=1.5, label='BVH (canal gains × 0.05)')
axes[1].plot(t_np,  hv_np, color='#555555', lw=1.0, ls='--', alpha=0.5, label='Head vel (ideal VOR)')
ax_fmt(axes[1], ylabel='−Eye vel (deg/s)')
axes[1].legend(fontsize=7)
axes[1].set_title('Eye velocity  — VOR abolished in BVH (oscillopsia during head motion)')

axes[2].plot(t_np, vs_net(st_bvh_h), color='#2166ac', lw=1.5, label='Healthy VS')
axes[2].plot(t_np, vs_net(st_bvh_p), color='#d6604d', lw=1.5, label='BVH VS')
ax_fmt(axes[2], ylabel='VS net (deg/s)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('Velocity storage — minimal charging in BVH')

axes[1].axvline(ROT_BVH, color='k', lw=0.7, ls='--', alpha=0.4)
axes[2].axvline(ROT_BVH, color='k', lw=0.7, ls='--', alpha=0.4)
fig.tight_layout()
plt.show()

---
## 4. Velocity Storage Dysfunction — Nodulus/Uvula Lesion

The cerebellar nodulus/uvula clamps velocity storage TC. Lesions produce:
- Prolonged OKAN (TC >> tau_vs) — VS no longer clamped
- Or in some accounts, shortened VS TC (nodulus ablation disrupts VS, not just its clamping)

Here we simulate **VS with tau_vs → 0.5 s** (near-abolished) vs healthy, showing
the effect on OKAN and post-rotatory VOR.

**Ref:** Cohen et al. (1981) *Brain Res*; Waespe et al. (1985) *Science* — nodulus ablation
prolongs OKAN (perflap: the nodulus normally SHORTENS TC by dumping VS). ⚠️

**Parameters:** `tau_vs → 0.5` (VS abolished), vs `tau_vs = 20` (healthy)

In [ ]:
%%time
THETA_NO_VS = with_brain(THETA, tau_vs=0.5, K_vs=0.001)

V_VS = 60.0; ROT_VS = 20.0; CST_VS = 60.0
t_vs, hv_vs = stim_mod.rotation_step(V_VS, rotate_dur=ROT_VS, coast_dur=CST_VS, dt=DT)
T_vs = len(t_vs)
t_vs_np = np.array(t_vs)

# Dark: VOR post-rotatory
st_vs_h = simulate(THETA,       t_vs, head_vel_array=hv_vs,
                   scene_present_array=jnp.zeros(T_vs),
                   target_present_array=jnp.zeros(T_vs),
                   max_steps=int((ROT_VS+CST_VS)/0.001)+500, return_states=True)
st_vs_n = simulate(THETA_NO_VS, t_vs, head_vel_array=hv_vs,
                   scene_present_array=jnp.zeros(T_vs),
                   target_present_array=jnp.zeros(T_vs),
                   max_steps=int((ROT_VS+CST_VS)/0.001)+500, return_states=True)

# OKN: visual only
V_OKN = 30.0; ON_OKN = 30.0; OFF_OKN = 50.0
t_okn = jnp.arange(0.0, ON_OKN+OFF_OKN, DT)
T_okn = len(t_okn)
v_sc  = jnp.zeros((T_okn, 3)).at[:, 0].set(jnp.where(t_okn < ON_OKN, V_OKN, 0.0))
sc_pr = jnp.where(t_okn < ON_OKN, 1.0, 0.0)
t_okn_np = np.array(t_okn)

st_okn_h = simulate(THETA,       t_okn, v_scene_array=v_sc, scene_present_array=sc_pr,
                    target_present_array=jnp.zeros(T_okn),
                    max_steps=int((ON_OKN+OFF_OKN)/0.001)+500, return_states=True)
st_okn_n = simulate(THETA_NO_VS, t_okn, v_scene_array=v_sc, scene_present_array=sc_pr,
                    target_present_array=jnp.zeros(T_okn),
                    max_steps=int((ON_OKN+OFF_OKN)/0.001)+500, return_states=True)

burst_h = extract_burst(st_okn_h, THETA)
burst_n = extract_burst(st_okn_n, THETA_NO_VS)
ev_okn_h = eye_vel(st_okn_h)
ev_okn_n = eye_vel(st_okn_n)
spv_h = slow_phase(t_okn_np, ev_okn_h, burst_h)
spv_n = slow_phase(t_okn_np, ev_okn_n, burst_n)
print('Done.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Velocity Storage Dysfunction — Nodulus/Uvula Lesion (tau_vs: 20s → 0.5s)',
             fontsize=10, fontweight='bold')

C_h = '#2166ac'; C_n = '#d6604d'

# VOR post-rotatory
for ax, states, label, c in [
    (axes[0,0], st_vs_h, f'Healthy (tau_vs={THETA.brain.tau_vs}s)',    C_h),
    (axes[0,0], st_vs_n, f'No VS   (tau_vs={THETA_NO_VS.brain.tau_vs}s)', C_n),
]:
    burst = extract_burst(states, THETA if label.startswith('H') else THETA_NO_VS)
    ev    = eye_vel(states)
    spv   = slow_phase(t_vs_np, ev, burst)
    ax.plot(t_vs_np, -spv, color=c, lw=1.5, label=label)
axes[0,0].axvline(ROT_VS, color='k', lw=0.7, ls='--', alpha=0.4)
ax_fmt(axes[0,0], ylabel='SPV (deg/s)', xlabel='Time (s)')
axes[0,0].legend(fontsize=7)
axes[0,0].set_title('Post-rotatory VOR in dark — TC shortened without VS')

# VS state VOR
axes[0,1].plot(t_vs_np, vs_net(st_vs_h), color=C_h, lw=1.5, label='Healthy')
axes[0,1].plot(t_vs_np, vs_net(st_vs_n), color=C_n, lw=1.5, label='No VS')
axes[0,1].axvline(ROT_VS, color='k', lw=0.7, ls='--', alpha=0.4)
ax_fmt(axes[0,1], ylabel='VS net (deg/s)', xlabel='Time (s)')
axes[0,1].legend(fontsize=7)
axes[0,1].set_title('VS state — healthy charges slowly; ablated decays fast')

# OKN + OKAN SPV
axes[1,0].plot(t_okn_np, spv_h, color=C_h, lw=1.5, label='Healthy')
axes[1,0].plot(t_okn_np, spv_n, color=C_n, lw=1.5, label='No VS')
axes[1,0].axvline(ON_OKN, color='k', lw=0.7, ls='--', alpha=0.4)
axes[1,0].axhline(V_OKN, color='gray', lw=0.7, ls=':', alpha=0.5)
ax_fmt(axes[1,0], ylabel='SPV (deg/s)', xlabel='Time (s)')
axes[1,0].legend(fontsize=7)
axes[1,0].set_title('OKN + OKAN — OKAN abolished without VS')

# VS state OKN
axes[1,1].plot(t_okn_np, -vs_net(st_okn_h), color=C_h, lw=1.5, label='Healthy')
axes[1,1].plot(t_okn_np, -vs_net(st_okn_n), color=C_n, lw=1.5, label='No VS')
axes[1,1].axvline(ON_OKN, color='k', lw=0.7, ls='--', alpha=0.4)
ax_fmt(axes[1,1], ylabel='VS net (deg/s)', xlabel='Time (s)')
axes[1,1].legend(fontsize=7)
axes[1,1].set_title('VS state during OKN — no charging without VS')

fig.tight_layout()
plt.show()

---
## 5. VOR Gain Adaptation (Cerebellar)

Prism or magnifying lens wear drives cerebellar VOR gain adaptation.  
In the model we approximate this by directly scaling `canal_gains`.

**Parameters:**
- Gain-up (2× lens): `canal_gains × 1.5`  
- Gain-down (0.5× lens): `canal_gains × 0.5`

**Expected:**
- Gain-up: VOR gain > 1 (over-compensates)
- Gain-down: VOR gain < 1 (under-compensates)
- Both: preserved VOR dynamics; only gain changes

**Ref:** Miles & Eighmy (1980) *J Neurophysiol*; Lisberger et al. (1994) ✅

In [ ]:
%%time
cg = np.array(THETA.sensory.canal_gains)
THETA_GAINUP   = with_sensory(THETA, canal_gains=tuple(cg * 1.5))
THETA_GAINDOWN = with_sensory(THETA, canal_gains=tuple(cg * 0.5))

# Sinusoidal VOR: 1 Hz ±30 deg/s
F_GA  = 1.0; V_GA  = 30.0; T_GA  = 8.0
t_ga  = jnp.arange(0.0, T_GA, DT)
T_ga  = len(t_ga)
hv_ga = jnp.array((V_GA * np.sin(2 * np.pi * F_GA * np.array(t_ga))).astype(np.float32))

ga_results = {}
for label, theta in [
    ('Healthy', THETA), ('Gain-up (×1.5)', THETA_GAINUP), ('Gain-down (×0.5)', THETA_GAINDOWN)
]:
    st = simulate(theta, t_ga,
                  head_vel_array       = hv_ga,
                  scene_present_array  = jnp.zeros(T_ga),
                  target_present_array = jnp.zeros(T_ga),
                  max_steps=int(T_GA/0.001)+500, return_states=True)
    ga_results[label] = dict(ev=eye_vel(st), ep=eye_pos(st))
    del st

t_ga_np  = np.array(t_ga)
hv_ga_np = np.array(hv_ga)
print('Done.')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle(f'VOR Gain Adaptation — Sinusoidal {F_GA:.0f} Hz ±{V_GA:.0f} deg/s',
             fontsize=10, fontweight='bold')

colors = {'Healthy': '#2166ac', 'Gain-up (×1.5)': '#1a9641', 'Gain-down (×0.5)': '#d6604d'}
for label, d in ga_results.items():
    axes[0].plot(t_ga_np, -d['ev'], color=colors[label], lw=1.5, label=label)
axes[0].plot(t_ga_np, hv_ga_np, color='#888888', lw=1.0, ls='--', label='Head vel')
ax_fmt(axes[0], ylabel='−Eye vel (deg/s)')
axes[0].legend(fontsize=7, loc='upper right', ncol=2)
axes[0].set_title('VOR response — same head velocity, different canal gains')

# 1-cycle zoom
t0_z = T_GA/2 - 1/F_GA; t1_z = T_GA/2 + 1/F_GA
msk = (t_ga_np >= t0_z) & (t_ga_np <= t1_z)
for label, d in ga_results.items():
    axes[1].plot(t_ga_np[msk], -d['ev'][msk], color=colors[label], lw=2.0, label=label)
axes[1].plot(t_ga_np[msk], hv_ga_np[msk], color='#888888', lw=1.0, ls='--', label='Head vel')
ax_fmt(axes[1], ylabel='−Eye vel (deg/s)', xlabel='Time (s)')
axes[1].legend(fontsize=7, loc='upper right')
axes[1].set_title('1-cycle zoom — steady state')

fig.tight_layout()
plt.show()

---
## 6. Caloric Equivalent — Asymmetric Canal Stimulation

Caloric irrigation warms/cools endolymph, creating a convection current that deflects the horizontal
cupula equivalently to slow sustained rotation.  
Model equivalent: brief sustained rotation of one "virtual" canal only.

A cleaner model approach: inject a direct canal-state perturbation on one side.

**Expected:**
- Warm irrigation (ipsilateral ampullofugal flow = same as ipsilateral rotation) → nystagmus beating ipsilateral
- Cold irrigation → nystagmus beating contralateral
- Mnemonic: COWS (Cold Opposite Warm Same)
- Duration ~90–120 s, declining (not sustained) — due to canal adaptation TC

**Approximate with:** very slow sustained rotation (1–2 deg/s) in one direction for 60–120 s

**Ref:** Bárány (1906); reviewed in Scherer et al. (2008) *J Vestib Res* ⚠️

In [ ]:
%%time
# Approximate caloric: slow rotation for 90 s (canal adapts over tau_c = 5s → decaying response)
V_CAL = 2.0; DUR_CAL = 120.0
t_cal = jnp.arange(0.0, DUR_CAL, DT)
T_cal = len(t_cal)
hv_cal = jnp.full(T_cal, V_CAL)   # constant slow rotation
t_cal_np = np.array(t_cal)

st_cal = simulate(THETA, t_cal,
                  head_vel_array       = hv_cal,
                  scene_present_array  = jnp.zeros(T_cal),
                  target_present_array = jnp.zeros(T_cal),
                  max_steps=int(DUR_CAL/0.001)+500, return_states=True)

ev_cal    = eye_vel(st_cal)
burst_cal = extract_burst(st_cal, THETA)
spv_cal   = slow_phase(t_cal_np, ev_cal, burst_cal)
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
fig.suptitle(f'Caloric Equivalent — Constant {V_CAL} deg/s Rotation ({DUR_CAL:.0f} s)',
             fontsize=10, fontweight='bold')

axes[0].axhline(V_CAL, color='#555555', lw=1.5, label=f'Head vel = {V_CAL} deg/s')
ax_fmt(axes[0], ylabel='Head vel (deg/s)')
axes[0].legend(fontsize=7)

axes[1].plot(t_cal_np, -spv_cal, color='#2166ac', lw=1.5, label='SPV')
axes[1].plot(t_cal_np, -ev_cal, color='#aaaaaa', lw=0.4, alpha=0.5, label='raw eye vel')
ax_fmt(axes[1], ylabel='SPV (deg/s)')
axes[1].legend(fontsize=7)
axes[1].set_title('Eye SPV — rises then decays as canals adapt (tau_c = %.0f s)' % THETA.sensory.tau_c)

axes[2].plot(t_cal_np, vs_net(st_cal), color='#35978f', lw=1.5, label='VS net')
ax_fmt(axes[2], ylabel='VS net (deg/s)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('Velocity storage — slow charging during sustained drive')

fig.tight_layout()
plt.show()

---
## 7. High-Acceleration Head Impulse (vHIT) — Main Sequence

Varies head impulse amplitude and measures VOR gain and catch-up saccade presence.

**Expected:**
- Gain ≈ 0.9–1.0 healthy for all amplitudes
- UVH: reduced gain ipsilesionally; overt catch-up saccade
- Catch-up saccade latency ~100–150 ms post-impulse

In [ ]:
%%time
VELOCITIES = [50, 100, 150, 200]   # deg/s peak

vhit_results = {'Healthy': {}, 'UVH left': {}}
for vp in VELOCITIES:
    for label, theta, direction in [
        ('Healthy',  THETA,     -1),   # leftward = toward lesioned side in UVH
        ('UVH left', THETA_UVH, -1),
    ]:
        t, hv1d = head_impulse_1d(direction=direction)
        # Scale peak velocity
        hv1d = hv1d * (vp / HIT_V)
        T = len(t)
        st = simulate(theta, t,
                      head_vel_array       = hv1d,
                      scene_present_array  = jnp.zeros(T),
                      target_present_array = jnp.zeros(T),
                      max_steps=int((REST_PRE+HIT_DUR+REST_POST)/0.001)+200,
                      return_states=True)
        ev = eye_vel(st)
        # Peak gain: ratio of peak eye vel to peak head vel
        peak_hv = float(np.abs(np.array(hv1d)).max())
        peak_ev = float(np.abs(ev).max())
        gain = peak_ev / peak_hv if peak_hv > 0 else 0
        vhit_results[label][vp] = dict(t=np.array(t), hv=np.array(hv1d), ev=ev, gain=gain)
        del st

print('Done.')
for vp in VELOCITIES:
    print(f'  {vp:3d} deg/s — Healthy gain: {vhit_results["Healthy"][vp]["gain"]:.2f}  '
          f'UVH gain: {vhit_results["UVH left"][vp]["gain"]:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('vHIT — Leftward Impulses (toward lesioned side)', fontsize=10, fontweight='bold')

C_h = '#2166ac'; C_u = '#d6604d'

for vp, ls in zip(VELOCITIES, ['-', '--', '-.', ':']):
    r_h = vhit_results['Healthy'][vp]
    r_u = vhit_results['UVH left'][vp]
    axes[0].plot(r_h['t'], -r_h['ev'], color=C_h, lw=1.2, ls=ls, label=f'{vp} deg/s')
    axes[0].plot(r_u['t'], -r_u['ev'], color=C_u, lw=1.2, ls=ls)

axes[0].plot([], [], color=C_h, lw=2, label='Healthy')
axes[0].plot([], [], color=C_u, lw=2, label='UVH left')
axes[0].axvline(REST_PRE, color='k', lw=0.6, ls='--', alpha=0.4)
axes[0].axvline(REST_PRE + HIT_DUR, color='k', lw=0.6, ls='--', alpha=0.4)
ax_fmt(axes[0], ylabel='−Eye vel (deg/s)', xlabel='Time (s)')
axes[0].legend(fontsize=7)
axes[0].set_title('Eye velocity during impulse')

# Gain vs peak head velocity
vps = VELOCITIES
gains_h = [vhit_results['Healthy'][v]['gain'] for v in vps]
gains_u = [vhit_results['UVH left'][v]['gain'] for v in vps]
axes[1].plot(vps, gains_h, 'o-', color=C_h, lw=2, ms=6, label='Healthy')
axes[1].plot(vps, gains_u, 'o-', color=C_u, lw=2, ms=6, label='UVH left')
axes[1].axhline(1.0, color='k', lw=0.8, ls='--', alpha=0.5, label='Ideal gain')
axes[1].set_xlabel('Peak head velocity (deg/s)', fontsize=8)
axes[1].set_ylabel('VOR gain (peak eye / peak head)', fontsize=8)
axes[1].set_ylim(0, 1.3)
axes[1].legend(fontsize=8)
axes[1].set_title('VOR gain vs amplitude')
axes[1].grid(True, alpha=0.2)

fig.tight_layout()
plt.show()

---
## Future: Conditions Requiring Model Extensions

### A. Spontaneous nystagmus from deafferentation — needs 30/70 afferent/intrinsic split
- Add `b_intrinsic` (70%) and `b_afferent` (30%) to BrainParams
- Per-population bias: `b_L = (b_intrinsic, b_intrinsic, b_intrinsic)`, `b_R = (b_intrinsic, ...)`
- Deafferentation: set `canal_gains_L → 0` AND `b_L = b_intrinsic` (afferent term lost)
- VN infarct: `b_L → 0` (total silencing)

### B. Alexander's law — nystagmus amplitude depends on gaze direction
- Nystagmus is larger when gaze is in direction of fast phase
- Requires NI-gated VS modulation or position-dependent efference copy
- Ref: Leigh & Zee (2015) *The Neurology of Eye Movements* ✅

### C. BPPV — canalithiasis / cupulolithiasis
- Otolith debris in semicircular canal → spurious angular velocity signal with head tilt
- Requires coupling from otolith (debris position) → canal afferent additional drive
- Brief (< 60 s, canalithiasis) or sustained (> 60 s, cupulolithiasis) response
- Geotropic vs apogeotropic patterns for horizontal canal BPPV
- Ref: Hall et al. (1979) *Ann Otol*; Epley (1992) *Otolaryngol Head Neck Surg* ✅

### D. Superior Canal Dehiscence (SCD)
- Bony defect creates a third window → abnormal sound/pressure sensitivity of superior canal
- Tullio phenomenon: sound-evoked nystagmus in the plane of the superior canal
- Requires canal-specific geometry and pressure-sensitivity term
- Ref: Minor et al. (1998) *Arch Otolaryngol Head Neck Surg* ✅

### E. Compensation timecourse — static vs dynamic
- Static compensation (resting nystagmus resolves): 2–7 days in primates
- Dynamic compensation (VOR gain recovery): weeks–months, cerebellar dependent
- Requires plasticity / learning rules on canal gains or VS weights
- Could be implemented as slow exponential recovery of `b_L` toward `b_vs`
- Ref: Curthoys & Halmagyi (1995); Smith & Curthoys (1989) ✅

### F. Menière's disease — episodic endolymphatic hydrops
- Sudden fluctuating gain changes + hearing loss
- Could be modeled as transient canal gain changes or cupula displacement

### G. Mal de Débarquement
- Persistent perception of motion after prolonged ship/boat exposure
- Possibly VS or cerebellum recalibrated to sustained slow oscillatory motion
- Could be modeled as adaptation of tau_vs or canal_gains to oscillatory stimulus